# Simplified Stages 5 & 6: LLM Filtering + ChromaDB Ingestion

This notebook provides a minimal, functional implementation of:
- **Stage 5**: LLM-based document filtering using question generation
- **Stage 6**: ChromaDB ingestion of filtered documents

**Requirements**: Chunked documents from stage 3 (`data/processed/documents_chunked.parquet`)

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import asyncio
import textwrap
from pathlib import Path
from typing import Dict, List, Tuple
import time
import polars as pl
import chromadb
from openai import AsyncOpenAI, OpenAI
from dotenv import load_dotenv
from tqdm.asyncio import tqdm
from pydantic import BaseModel, Field
import langchain_openai
from loguru import logger
from bs4 import BeautifulSoup
from functools import reduce
from itertools import product
import sys
sys.path.append('..')
from notebook_checkpoint_integration import list_experiments
from improved_checkpoint_system import SafeCheckpointManager
from loguru_disk_setup import setup_notebook_logging, ExperimentPhaseLogger
log_info = setup_notebook_logging()
# Load environment variables
load_dotenv(override=True)

# Configuration
SAMPLE_SIZE = 100  # Limit for testing - set to None for full dataset
BATCH_SIZE = 10     # Number of chunks to process in parallel
MAX_CONCURRENT = 7 # Maximum concurrent API requests

# Paths
INPUT_FILE = Path("../data/processed/nq_question_answer_chunked_embeddings_with_suffix.parquet")
RAW_QA_DATA = Path('../data/processed/nq_question_answer.parquet')
CLEANED_NON_CHUNKED = Path('../data/processed/nq_question_answer_cleaned.parquet')
OUTPUT_FILE = Path('../data/processed/wikitext_approved_questions.json')
CHROMA_PATH = Path("../chroma_wikitext")
client = OpenAI(base_url=os.getenv('ORQ_BASE_URL'), api_key=os.getenv("ORQ_API_KEY"))
client_async = AsyncOpenAI(base_url=os.getenv('ORQ_BASE_URL'), api_key=os.getenv("ORQ_API_KEY"))

print(f"✅ Setup complete")
print(f"📂 Input: {INPUT_FILE}")
print(f"📂 Output: {OUTPUT_FILE}")
print(f"📂 Raw QA Data: {RAW_QA_DATA}")
print(f"📂 Cleaned Non-Chunked Data: {CLEANED_NON_CHUNKED}")
print(f"🗃️  ChromaDB: {CHROMA_PATH}")
print(f"ORQ API KEY: {os.getenv('ORQ_API_KEY')[-5:]}")

html_cleaned_df = pl.scan_parquet(CLEANED_NON_CHUNKED)

2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_experiment_logging:105 - Logging initialized for wikiqa_notebook
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_experiment_logging:106 - Log directory: /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/nbs/logs/wikiqa_experiments
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_experiment_logging:107 - Log level: INFO
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_experiment_logging:108 - Console logging: True
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_notebook_logging:194 - ============================================================
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_notebook_logging:195 - WikiQA Experiment Session Started
2025-09-20 15:51:22 | INFO     | loguru_disk_setup:setup_notebook_logging:196 - ============================================================


✅ Improved 400 error handling module created!

📝 Key functions provided:
  - llm_call_with_error_tracking(): Enhanced LLM call with structured error returns
  - process_question_with_error_handling(): Process results with error metadata
  - summarize_error_statistics(): Generate error statistics from results

🔧 See INTEGRATION_EXAMPLE variable for usage in notebook
✅ Fixed checkpoint integration loaded!
🔧 Usage:
   # Test the system:
   test_results = await test_checkpoint_system()
   
   # Run single experiment:
   results = await run_single_experiment_improved(
       qa_pairs=filtering_results,
       cleaned_df=html_cleaned_df,
       k=20, retrieval_kind='enhanced_rag', rerank=True,
       n_questions=200, force_new=False
   )
   
   # Run full experiment suite:
   all_results = await run_full_experiment_improved(
       clean_df=html_cleaned_df,
       k_range=[1, 5, 10, 20, 50, 100],
       n_questions=200, parallel=3, force_new=False
   )
   
   # Monitor experiments:
   list_e

### Create cleaned version of the dataframe

In [ ]:
# pl.scan_parquet('../data/processed/nq_question_answer.parquet').with_columns(
#         pl.col('document_html').map_elements(lambda x: BeautifulSoup(x, "html.parser").get_text(), return_dtype=pl.String).alias('document_text')
# ).sink_parquet('../data/processed/nq_question_answer_cleaned.parquet')

## Stage 5: LLM Filtering

Simple implementation of question-based filtering:
1. Group consecutive chunks (3 chunks per group)
2. Generate a question from combined text
3. Evaluate if question seems human-like
4. Keep chunks that pass evaluation

In [ ]:
class SimpleFilter:
    def __init__(self):
        self.client = client_async
        self.model = "openai/gpt-4.1"

    async def generate_question(self, text: str) -> tuple[bool, str, str, str | None, str | None]:
        """Generate a question from text and provide reasoning."""
        prompt = textwrap.dedent(f"""Generate ONE specific question that a human might ask about this text.

Format:
{{
    "question": "[question]"
    "reasoning": "[why this question is appropriate]"
}}
Text:
{text[:2000]}""")

        class AnswerModel(BaseModel):
            question: str = Field(..., description="The generated question")
            answer: str = Field(..., description="The answer to the question based on the text")
            reasoning: str = Field(..., description="The reasoning as to why this is a good questions")
            probability_human: float =  Field(..., description="0.0-1.0, how likely is it that a human would ask this question")
            main_chunk_id: str = Field(..., description="The unique_id of the chunk that is most relevant to this question")

        try:
            response = await self.client.responses.parse(
                model=self.model,
                input=[{"role": "user", "content": prompt}],
                text_format=AnswerModel,
            )
            response = response.output_parsed
            return True, response.question, response.reasoning, response.probability_human, response.main_chunk_id, response.answer

        except Exception as e:
            logger.exception(f"Error during question generation: {e}")
            return False, "", f"Generation failed: {str(e)}", None, None, None

    def group_consecutive_chunks(self, df: pl.DataFrame, chunk_count: int = 3, min_chunks: int = 100) -> List[Dict]:
        """Group consecutive chunks for processing."""
        groups = []
        for name, data in df.group_by('document_url'):
            if data.height < min_chunks:
                continue
            documents = data.to_dicts()

            for i in range(0, len(documents), chunk_count):
                end_idx = min(i + chunk_count, len(documents))
                group_docs = documents[i:end_idx]
                groups.append(
                    {
                        "docs": group_docs,
                        "combined_text": "\n\n".join([f'{doc["unique_id"]=} | {doc["collection_suffix"]}\n{doc["chunked_prompt"]=}' for doc in group_docs]),
                        "ids": [doc["unique_id"] for doc in group_docs],
                        'collection_suffix': documents[0]['collection_suffix'],
                        'document_url': name[0]
                    }
                )

        return groups

    async def process_group(self, group: Dict) -> Dict[str, Dict]:
        """Process a group of chunks through question generation and evaluation."""
        combined_text = group["combined_text"]
        chunk_ids = group["ids"]
        collection_suffix = group['collection_suffix']
        document_url = group['document_url']

        # Generate question
        status, question, q_reasoning, probability_human, main_chunk_id, answer = await self.generate_question(combined_text)

        if not status:
            # If question generation fails, mark all chunks as failed
            return {
                    "probability_human": 0.0,
                    "generated_question": "",
                    "question_reasoning": q_reasoning,
                    "main_chunk_id": None,
                    "status": "question_generation_failed",
                    "answer": None,
                    'document_url': document_url,
                    'collection_suffix': collection_suffix,
            }

        # All chunks in group get same result
        return {
                "probability_human": probability_human,
                "generated_question": question,
                "question_reasoning": q_reasoning,
                "main_chunk_id": main_chunk_id,
                "answer": answer,
                "status": "question_generation_succeeded",
                'document_url': document_url,
                'collection_suffix': collection_suffix,
            }


print("🔧 SimpleFilter class defined")
filter = SimpleFilter()
chunked_data = pl.scan_parquet(INPUT_FILE)

# await filter.generate_question("This is a test text to generate a question from.")
gs = filter.group_consecutive_chunks(chunked_data.limit(100).collect(), chunk_count=3, min_chunks=20)
await filter.process_group(gs[0])

🔧 SimpleFilter class defined


{'probability_human': 0.95,
 'generated_question': 'What was the impact of European colonisation on Indigenous Australians between 1788 and 1850?',
 'question_reasoning': "This question is appropriate because the text explicitly mentions that 'European colonisation would have a devastating effect on the pre-existing population of Indigenous Australians,' indicating that this is a significant aspect of this historical period and likely to be of high interest to readers.",
 'main_chunk_id': '203ca45d-e4fc-4666-912c-61ff83b8f960',
 'answer': '',
 'status': 'question_generation_succeeded',
 'document_url': 'https://en.wikipedia.org//w/index.php?title=History_of_Australia_(1788%E2%80%931850)&amp;oldid=819687010',
 'collection_suffix': 'History_of_Australia_1788E280931850'}

In [19]:
# Load and sample data
print("📥 Loading chunked documents...")
df = pl.read_parquet(INPUT_FILE)
print(f"📊 Loaded {len(df)} documents")

# if SAMPLE_SIZE:
#     df = df.head(1000)
#     print(f"🎯 Using sample of {len(df)} documents")

# Initialize filter
filter_engine = SimpleFilter()
print("✅ Filter initialized")

📥 Loading chunked documents...
📊 Loaded 212044 documents
✅ Filter initialized


In [23]:
document_urls = df.group_by('document_url').count().filter(pl.col('count') > 100)['document_url'].to_list()
document_urls

/var/folders/bd/lg0gz3wd5jgdrhj47_f2nfyh0000gn/T/ipykernel_2099/1368443298.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  document_urls = df.group_by('document_url').count().filter(pl.col('count') > 100)['document_url'].to_list()


['https://en.wikipedia.org//w/index.php?title=2018_NCAA_Division_I_Men%27s_Basketball_Tournament&amp;oldid=837560621',
 'https://en.wikipedia.org//w/index.php?title=Singapore&amp;oldid=838168836',
 'https://en.wikipedia.org//w/index.php?title=Beijing&amp;oldid=838137684',
 'https://en.wikipedia.org//w/index.php?title=Atomic_bombings_of_Hiroshima_and_Nagasaki&amp;oldid=838327110',
 'https://en.wikipedia.org//w/index.php?title=Rafael_Nadal&amp;oldid=838533145',
 'https://en.wikipedia.org//w/index.php?title=Bay_of_Pigs_Invasion&amp;oldid=817526436',
 'https://en.wikipedia.org//w/index.php?title=History_of_television&amp;oldid=838235565',
 'https://en.wikipedia.org//w/index.php?title=Toronto_Maple_Leafs&amp;oldid=807290929',
 'https://en.wikipedia.org//w/index.php?title=The_Phantom_of_the_Opera_(1986_musical)&amp;oldid=838349429',
 'https://en.wikipedia.org//w/index.php?title=Air_pollution&amp;oldid=822370127',
 'https://en.wikipedia.org//w/index.php?title=Guantanamo_Bay_detention_camp&amp

In [25]:
# Process filtering
async def run_filtering(df, min_chunks: int = 100, desired_nr_questions: int = 200) -> list[dict]:
    print("🔍 Starting LLM filtering...")

    # Group consecutive chunks
    groups = filter_engine.group_consecutive_chunks(df, chunk_count=3, min_chunks=min_chunks)
    print(f"📦 Created {len(groups)} chunk groups")

    # Process groups with concurrency control
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def process_with_semaphore(group):
        async with semaphore:
            return await filter_engine.process_group(group)

    # Process all groups
    results = []
    for i in tqdm(range(0, len(groups), BATCH_SIZE), desc="Processing batches"):
        batch_groups = groups[i : i + BATCH_SIZE]
        batch_tasks = [process_with_semaphore(group) for group in batch_groups]
        batch_results = await asyncio.gather(*batch_tasks, return_exceptions=True)

        for result in batch_results:
            if isinstance(result, Exception):
                print(f"⚠️ Error in batch: {result}")
            else:
                results.append(result)
        if len(results) >= desired_nr_questions:
            break

    # Combine all results
    # all_results = {}
    # for result in results:
    #     all_results.update(result)

    return results


# Run the filtering
filtering_results = await run_filtering(df, min_chunks=100, desired_nr_questions=500)
print(f"✅ Filtering complete: {len(filtering_results)} chunks processed")

🔍 Starting LLM filtering...
📦 Created 13571 chunk groups


Processing batches:   4%|▎         | 49/1358 [04:49<2:09:04,  5.92s/it]


✅ Filtering complete: 500 chunks processed


In [26]:
# Filter and save results
print("📊 Processing filtering results...")

# Get chunks that passed filtering
passed_ids = [result['main_chunk_id'] for result in filtering_results 
              if result.get("probability_human", 0.0) > 0.8]

print(f"✅ {len(passed_ids)} chunks passed filtering (out of {len(filtering_results)})")
print(f"📊 Success rate: {len(passed_ids)/len(filtering_results)*100:.1f}%")

# Filter dataframe to only include passed chunks
filtered_df = df.filter(pl.col("unique_id").is_in(passed_ids))

# Add filtering metadata
metadata_rows = []
for result in filtering_results:
    metadata_rows.append({
        "unique_id": result['main_chunk_id'],
        "human_would_ask": result.get("human_would_ask", False),
        "generated_question": result.get("generated_question", ""),
        "question_reasoning": result.get("question_reasoning", ""),
        "evaluation_reasoning": result.get("evaluation_reasoning", ""),
        "main_chunk_id": result.get("main_chunk_id", None),
        'document_url': result.get('document_url', ''),
        'collection_suffix': result.get('collection_suffix', ''),
        'answer': result.get('answer', ''),
        "status": result.get("status", "unknown"),
    })

metadata_df = pl.DataFrame(metadata_rows)

# Join with original data
final_df = filtered_df.join(metadata_df, on="unique_id", how="left")

# Save filtered results
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
final_df.write_parquet(OUTPUT_FILE)

print(f"💾 Saved {len(final_df)} filtered documents to {OUTPUT_FILE}")
print(f"📝 Columns: {final_df.columns}")

📊 Processing filtering results...
✅ 396 chunks passed filtering (out of 500)
📊 Success rate: 79.2%
💾 Saved 395 filtered documents to ../data/processed/wikitext_approved_questions.json
📝 Columns: ['chunked_prompt', 'document_url', 'context_length', 'token_count', 'unique_id', 'embedding', 'collection_suffix', 'human_would_ask', 'generated_question', 'question_reasoning', 'evaluation_reasoning', 'main_chunk_id', 'document_url_right', 'collection_suffix_right', 'answer', 'status']


In [27]:
json.dump(filtering_results, open('../data/processed/wikitext_approved_questions.json', 'w+'), indent=4)

In [3]:
import json
filtering_results = json.load(open('../data/processed/wikitext_approved_questions.json'))

In [4]:
ids = [r['main_chunk_id'] for r in filtering_results if r['probability_human'] > 0.8]
print(f"✅ Loaded {len(ids)} approved question chunk IDs")

✅ Loaded 396 approved question chunk IDs


## Stage 6: ChromaDB Ingestion

Simple ChromaDB ingestion of filtered documents with OpenAI embeddings.

## Test ChromaDB Query

In [5]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collections = chroma_client.list_collections()
from chromadb import Collection
collections
test_collection = chroma_client.get_collection("wikiqa_The_Crossing_TV_series")

In [6]:
from context_is_king.evaluation.judge_evaluator import JudgeEvaluator
from context_is_king.models.interface import ModelInterface
from context_is_king.rag_pipeline.query_enhancement.dual_retrieval import DualRetrieval
from context_is_king.rag_pipeline.query_enhancement.query_rewriter import QueryRewriter
from context_is_king.rag_pipeline.reranking.module import RerankerModule
from context_is_king.rag_pipeline.retrieval.engine import RetrievalEngine
from context_is_king.rag_pipeline.types import PipelineConfig

model_interface = ModelInterface()
config = PipelineConfig(quiet_mode=True)
judge = JudgeEvaluator(model_interface=model_interface, judge_model=config.judge_model)
query_rewriter = QueryRewriter(pipeline_config=config)
reranker = RerankerModule(config=config)
retrieval_engine = RetrievalEngine(config=config)

dual_retrieval = DualRetrieval(retrieval_engine=retrieval_engine, query_rewriter=query_rewriter, pipeline_config=config)
EMBEDDING_DIMENSION = 1536

🔗 Initialized Model Interface
📡 API Base URL: https://api.orq.ai/v2/proxy
🤖 Available Models: 14
🔄 Rate Limiting: 5 retries, 1.0s base delay


JudgeEvaluator initialized with model: gpt-4.1

🔗 Initialized Model Interface
📡 API Base URL: https://api.orq.ai/v2/proxy
🤖 Available Models: 14
🔄 Rate Limiting: 5 retries, 1.0s base delay


QueryRewriter initialized

MPS GPU detected and available

Initializing reranker model on mps

RerankerModule initialized with cross-encoder/ms-marco-MiniLM-L-6-v2 on mps

RetrievalEngine initialized

🔗 Initialized Model Interface
📡 API Base URL: https://api.orq.ai/v2/proxy
🤖 Available Models: 14
🔄 Rate Limiting: 5 retries, 1.0s base delay


DualRetrieval initialized

In [7]:
import random

import openai
from openai import OpenAI

from context_is_king.rag_pipeline.retrieval.engine import extract_retry_delay

openai_client = OpenAI(
    base_url=os.getenv("ORQ_BASE_URL"),
    api_key=os.getenv("ORQ_API_KEY"),
)
# openai_client.embeddings.create(input='test', model='azure/text-embedding-3-small')

# Test a simple query
test_query = "What is machine learning?"
print(f"🔍 Testing query: '{test_query}'")


async def base_retrieval(
    query: str,
    collection: Collection,
    k: int = 5,
    dimensions: int = EMBEDDING_DIMENSION,
    retries: int = 4,
    *,
    shuffle: bool = False,
):
    # Generate embedding for query
    query_embedding = None
    for i in range(retries):
        try:
            query_response = await client_async.embeddings.create(
                model="azure/text-embedding-3-small",
                input=[query],
                dimensions=dimensions,
            )
            query_embedding = query_response.data[0].embedding
            break
        except openai.RateLimitError as e:
            sleep_duration = extract_retry_delay(error_message=str(e))
            logger.error(
                f"Judge evaluation attempt {i + 1} failed. Sleeping for {sleep_duration} seconds: {type(e)} - {e}"
            )
            await asyncio.sleep(sleep_duration + 1 + random.uniform(1, 10))  # Exponential backoff
        except Exception as e:
            if "400" in str(e):
                logger.error(f"Bad request error, not retrying: {e}")
                return []
            sleep_duration = 2**i
            logger.error(f"llm call iteration {i} failed. Sleeping for {sleep_duration} type={type(e)}: {e}")
            await asyncio.sleep(sleep_duration + random.uniform(1, 10))
            continue
    if query_embedding is None:
        logger.error(f'Failed to get embedding after {retries} retries')
        return []

    # Query ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding], n_results=k, include=["documents", "metadatas", "distances"]
    )
    formatted_results = retrieval_engine._convert_to_retrieval_results(chroma_results=results, query=query)
    if shuffle:
        random.shuffle(formatted_results)

    return formatted_results


results = await base_retrieval(test_query, collection=test_collection, k=3, shuffle=True)

print(results)

🔍 Testing query: 'What is machine learning?'
[RetrievalResult(chunk=DocumentChunk(content='Tools\n\nWhat links hereRelated changesUpload fileSpecial pagesPermanent linkPage informationWikidata itemCite this page \n\nPrint/export\n\nCreate a bookDownload as PDFPrintable version \n\nLanguages\n\nČeštinaFrançaisItalianoРусский \nEdit links \n\n This page was last edited on 27 April 2018, at 07:56.\nText is available under the Creative Commons Attribution-ShareAlike License;\nadditional terms may apply.  By using this site, you agree to the Terms of Use and Privacy Policy. Wikipedia® is a registered trademark of the Wikimedia Foundation, Inc., a non-profit organization.\n\nPrivacy policy\nAbout Wikipedia\nDisclaimers\nContact Wikipedia\nDevelopers\nCookie statement\nMobile view\nEnable previews', doc_id='', chunk_id='286b4c85-8b81-49f2-8e2c-66c8acd70b81', start_char=0, end_char=0, metadata={'document_url': 'https://en.wikipedia.org//w/index.php?title=The_Crossing_(TV_series)&amp;oldid=8384

In [8]:
results = await dual_retrieval.retrieve_dual("What is machifne learning?", k=3, collection=test_collection, dimensions=EMBEDDING_DIMENSION)
results.model_dump().keys() 

dict_keys(['original_query', 'rewritten_query', 'original_results', 'rewritten_results', 'combined_results', 'fusion_metadata', 'total_time_ms'])

In [9]:
from context_is_king.rag_pipeline.types import RetrievalResult

def create_rag_prompt(query: str, retrieved_contexts: list[RetrievalResult] | str) -> tuple[str, str]:

    # Format retrieved contexts
    context_section = ""
    if not isinstance(retrieved_contexts, str):
        for i, context in enumerate(retrieved_contexts, 1):
            context_section += f"<chunk {i}>\n{context.chunk.content.strip()}\n<chunk {i}>\n\n"
    else:
        context_section = retrieved_contexts

    rag_prompt_template = textwrap.dedent(f"""
You are a helpful assistant that answers questions based on the provided context information.

<context>
{context_section}
</context>

INSTRUCTIONS:
- Answer the question using ONLY the information provided in the contexts above
- If the contexts don't contain enough information to answer the question, say "I don't have enough information in the provided contexts to answer this question"
- Be specific and cite which context(s) you're drawing information from
- If contexts contradict each other, mention this explicitly
- Keep your answer concise but complete

QUESTION: {query}

ANSWER:""")
    return rag_prompt_template, context_section


print(create_rag_prompt("What is machine learning?", results.combined_results[:3])[:200])


('\nYou are a helpful assistant that answers questions based on the provided context information.\n\n<context>\n<chunk 1>\nTools\n\nWhat links hereRelated changesUpload fileSpecial pagesPermanent linkPage informationWikidata itemCite this page \n\nPrint/export\n\nCreate a bookDownload as PDFPrintable version \n\nLanguages\n\nČeštinaFrançaisItalianoРусский \nEdit links \n\n This page was last edited on 27 April 2018, at 07:56.\nText is available under the Creative Commons Attribution-ShareAlike License;\nadditional terms may apply.  By using this site, you agree to the Terms of Use and Privacy Policy. Wikipedia® is a registered trademark of the Wikimedia Foundation, Inc., a non-profit organization.\n\nPrivacy policy\nAbout Wikipedia\nDisclaimers\nContact Wikipedia\nDevelopers\nCookie statement\nMobile view\nEnable previews\n<chunk 1>\n\n<chunk 2>\nExternal links[edit]\n\nOfficial website\nThe Crossing on IMDb \n\n[hide]\n\nv\nt\ne\n\nABC programming (current and upcoming)\n\nPrimetime\n

In [10]:
from typing import Any
import openai

from context_is_king.evaluation.judge_evaluator import extract_retry_delay
from improved_400_error_handling import process_question_with_error_handling, summarize_error_statistics


async def llm_call_with_error_tracking(
    prompt: str,
    client_async,
    model_interface,
    model: str = "gpt-4.1-mini",
    max_tokens: int = 1000,
    temperature: float = 0.0,
    response_model: type[BaseModel] | None = None,
    retries: int = 3,
    timeout: int = 30,
) -> tuple[str | dict[str, Any] | BaseModel, int]:
    """
    Enhanced LLM call that returns structured error information for 400 errors.

    Returns:
        Tuple of (response, token_count) where response can be:
        - String: Successful text response
        - BaseModel: Successful structured response
        - Dict: Error information with keys:
            - error: Error type (e.g., "content_policy", "rate_limit", "unknown")
            - message: Full error message
            - code: HTTP error code (e.g., 400)
            - generated_answer: None (for compatibility)
    """
    for i in range(retries):
        try:
            if response_model:
                completion = await client_async.chat.completions.parse(
                    messages=[{"role": "user", "content": prompt}],
                    response_format=response_model,
                    model=model_interface.get_model_config(model).api_name,
                    temperature=temperature,
                    timeout=timeout,
                )
                message = completion.choices[0].message
                if message.parsed:
                    return message.parsed, completion.usage.prompt_tokens
            else:
                response = await client_async.chat.completions.create(
                    model=model_interface.get_model_config(model).api_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                    temperature=temperature,
                    timeout=timeout,
                )
                msg = response.choices[0].message.content
                if hasattr(response, "usage") and hasattr(response.usage, "prompt_tokens"):
                    return msg, response.usage.prompt_tokens
                return msg, 0

        except openai.RateLimitError as e:  # noqa: PERF203
            # Try with fallback model for rate limits
            if i == 0 and not response_model:  # Only try fallback once
                try:
                    response = await llm_call_with_error_tracking(
                        prompt=prompt,
                        client_async=client_async,
                        model_interface=model_interface,
                        model="gemini-2.5-flash",
                        retries=1,
                        timeout=timeout,
                        response_model=response_model,
                        temperature=temperature,
                        max_tokens=max_tokens,
                    )
                    return response
                except Exception:  # noqa: S110
                    pass  # Continue with regular retry logic

            sleep_duration = extract_retry_delay(error_message=str(e))
            logger.error(f"LLM Call attempt {i + 1} failed. Sleeping for {sleep_duration} seconds: {type(e)} - {e}")
            await asyncio.sleep(sleep_duration + 1)

        except Exception as e:
            error_str = str(e)

            # Handle 400 errors (content policy violations)
            if "400" in error_str:
                logger.error(f"Bad request error (content policy), not retrying: {e}")

                # Parse error details if possible
                error_info = {
                    "error": "content_policy",
                    "message": error_str,
                    "code": 400,
                    "generated_answer": None,
                }

                # Try to extract more specific error message
                if "content management policy" in error_str or "content filtering" in error_str:
                    error_info["error"] = "content_policy"
                elif "invalid request" in error_str:
                    error_info["error"] = "invalid_request"
                else:
                    error_info["error"] = "bad_request"

                return error_info, 0

            # Other errors - retry with exponential backoff
            sleep_duration = 2**i
            logger.error(f"llm call iteration {i} failed. Sleeping for {sleep_duration} type={type(e)}: {e}")
            await asyncio.sleep(sleep_duration)

    # All retries exhausted
    logger.error(f"All {retries} retries exhausted for LLM call")
    return {
        "error": "max_retries_exceeded",
        "message": f"Failed after {retries} attempts",
        "code": None,
        "generated_answer": None,
    }, 0


# async def llm_call(
#     prompt: str,
#     model: str = "gpt-4.1-mini",
#     max_tokens: int = 1000,
#     temperature: float = 0.0,
#     response_model: type[BaseModel] | None = None,
#     retries: int = 3,
#     timeout: int = 30,
# ):
#     for i in range(retries):
#         try:
#             if response_model:
#                 completion = await client_async.chat.completions.parse(
#                     messages=[{"role": "user", "content": prompt}],
#                     response_format=response_model,
#                     model=model_interface.get_model_config(model).api_name,
#                     temperature=temperature,
#                     timeout=timeout,
#                 )
#                 message = completion.choices[0].message
#                 if message.parsed:
#                     return message.parsed, completion.usage.prompt_tokens
#             else:
#                 response = await client_async.chat.completions.create(
#                     model=model_interface.get_model_config(model).api_name,
#                     messages=[
#                         # {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context information."},
#                         {"role": "user", "content": prompt}
#                     ],
#                     max_tokens=max_tokens,
#                     temperature=temperature,
#                     timeout=timeout,
#                 )
#                 msg = response.choices[0].message.content
#                 if hasattr(response, "usage") and hasattr(response.usage, "prompt_tokens"):
#                     return msg, response.usage.prompt_tokens
#                 return msg, 0
#         except openai.RateLimitError as e:  # noqa: PERF203
#             if model == 'gpt-4.1-mini' and i == 1:
#                 logger.info('Retrying call with gemini-2.5-flash')
#                 response = llm_call(
#                     prompt=prompt,
#                     model="gemini-2.5-flash",
#                     retries=1,
#                     timeout=timeout,
#                     response_model=response_model,
#                     temperature=temperature,
#                     max_tokens=max_tokens,
#                 )
#             sleep_duration = extract_retry_delay(error_message=str(e))
#             logger.error(f"LLM Call attempt {i + 1} failed. Sleeping for {sleep_duration} seconds: {type(e)} - {e}")
#             await asyncio.sleep(sleep_duration + 1)  # Exponential backoff
#         except Exception as e:
#             if "400" in str(e):
#                 logger.error(f"Bad request error, not retrying: {e}")
#                 return "", 0
#             sleep_duration = 2**i
#             logger.error(f"llm call iteration {i} failed. Sleeping for {sleep_duration} type={type(e)}: {e}")
#             await asyncio.sleep(sleep_duration)
#     return "", 0


# await llm_call("hi whats you name?", model="gpt-4.1")


class AnswerModel(BaseModel):
    test_field: str = Field(..., description="Just a test field")


# await llm_call("hi whats you name?", model="gpt-4.1-mini", response_model=AnswerModel, retries=1, timeout=5)

await llm_call_with_error_tracking(
    "hi give me a sheep riddle",
    model="gpt-4.1-mini",
    response_model=AnswerModel,
    retries=1,
    timeout=5,
    client_async=client_async,
    model_interface=model_interface,
)

(AnswerModel(test_field="Sure! Here's a sheep riddle for you:\n\nI wander in pastures, fluffy and white,\nCounting me all day, out of sight.\nWith wool to shear and a soft bleat sound,\nWhat am I that's always around?"),
 62)

In [11]:

def extract_context_from_stage_output(stage_output):
    """Extract context text from stage output data"""
    # Handle pandas Series/NaN values properly
    if stage_output is None or stage_output == "":
        return ""

    try:
        # Parse the stage output if it's a string
        if isinstance(stage_output, str):
            stage_data = json.loads(stage_output)
        else:
            stage_data = stage_output

        # Extract content from chunks
        context_texts = []
        if isinstance(stage_data, list):
            for item in stage_data:
                content = item.chunk.content if hasattr(item, 'chunk') and hasattr(item.chunk, 'content') else None
                if content:  # Only add non-empty content
                    context_texts.append(str(content))

        return "\n".join(context_texts)
    except (json.JSONDecodeError, AttributeError, KeyError, TypeError):
        # If parsing fails, try to return the raw content as string
        try:
            return str(stage_output) if stage_output else ""
        except:
            return ""


In [12]:
from json import JSONDecodeError
from tqdm.auto import tqdm
from typing import Any
import tiktoken
import aiofiles
from collections.abc import MutableMapping

tokenizer = tiktoken.encoding_for_model("gpt-4o")


def dump_pydantic_models(obj: Any) -> Any:
    """
    Recursively converts any Pydantic models found in nested structures to dictionaries.

    Args:
        obj: Can be a dict, list, Pydantic model, or any other type

    Returns:
        The same structure but with all Pydantic models converted to dicts
    """
    if isinstance(obj, BaseModel):
        return obj.model_dump()

    elif isinstance(obj, dict):
        return {key: dump_pydantic_models(value) for key, value in obj.items()}

    elif isinstance(obj, list):
        return [dump_pydantic_models(item) for item in obj]

    elif isinstance(obj, tuple):
        return tuple(dump_pydantic_models(item) for item in obj)

    elif isinstance(obj, set):
        return {dump_pydantic_models(item) for item in obj}

    else:
        return obj


def count_tokens_approximate(text: str, tokenizer):
    """Approximate token count using tiktoken (GPT-4 tokenizer)"""
    if text is None or text == "":
        return 0

    try:
        # Use GPT-4 tokenizer for approximation
        tokens = tokenizer.encode(str(text))
        return len(tokens)
    except Exception:
        # Fallback: rough approximation (4 chars per token on average)
        text_str = str(text) if text else ""
        return max(1, len(text_str) // 4) if text_str else 0


def flatten(dictionary: dict, parent_key: str = "", separator: str = "_"):
    items = []
    for key, value in dictionary.items():
        new_key = parent_key + separator + key if parent_key else key
        if isinstance(value, MutableMapping):
            items.extend(flatten(value, new_key, separator=separator).items())
        else:
            items.append((new_key, value))
    return dict(items)


# async def run_single_experiment_improved(
#     qa_pairs,
#     cleaned_df,  # Your polars LazyFrame
#     k: int = 3,
#     iterations_per_question: int = 3,
#     retrieval_kind: str = "basic_rag",
#     *,
#     rerank: bool = True,
#     n_questions: int | None = None,
#     force_new: bool = False,
#     retries: int = 3,
# ):
#     """
#     Improved version of your run_single_experiment function with safe checkpointing.
#     Uses existing notebook variables: chroma_client, tokenizer, etc.
#     """

#     checkpoint_manager = SafeCheckpointManager()
#     n_questions = n_questions or len(qa_pairs)

#     # Start or resume experiment safely
#     experiment_id, is_resume, is_completed = await checkpoint_manager.start_experiment(
#         k=k, retrieval_kind=retrieval_kind, rerank=rerank, n_questions=n_questions, force_new=force_new
#     )

#     # If experiment is already completed, return cached results immediately
#     if is_completed:
#         logger.info(f"Experiment {experiment_id} already completed, loading cached results")
#         return await checkpoint_manager.load_completed_experiment_results(experiment_id)

#     if is_resume:
#         # Get completed questions to skip
#         completed_questions = await checkpoint_manager.get_completed_questions(experiment_id)
#         logger.info(f"Resuming experiment {experiment_id}: {len(completed_questions)} questions already completed")
#     else:
#         completed_questions = set()
#         logger.info(f"Starting new experiment {experiment_id}")

#     try:
#         # Create progress bar for questions
#         questions_to_process = [
#             (q_index, doc)
#             for q_index, doc in enumerate(qa_pairs)
#             if q_index not in completed_questions and doc.get("probability_human", 0) >= 0.8
#         ][:n_questions]

#         logger.info(f"Processing {len(questions_to_process)} questions (skipping {len(completed_questions)} completed)")

#         # Process questions with checkpoint recovery and progress bar
#         progress_bar = tqdm(
#             questions_to_process, desc=f"Experiment {experiment_id[:16]}...", unit="questions", disable=False
#         )

#         for q_index, doc in progress_bar:
#             # Update progress bar description with current question
#             progress_bar.set_postfix({"q_idx": q_index, "retrieval": retrieval_kind, "k": k, "rerank": rerank})

#             # Get collection for this document
#             collection = chroma_client.get_collection(f"wikiqa_{doc['collection_suffix']}")
#             if collection.count() == 0:
#                 logger.warning(f"Collection {collection.name} is empty, skipping question {q_index}")
#                 continue

#             question = doc["generated_question"]
#             ground_truth = doc["answer"]
#             ground_truth_chunk_id = doc["main_chunk_id"]

#             # Your existing experiment logic here
#             question_results = []

#             # Process this question with your existing logic
#             is_full_context = retrieval_kind == "full_context"

#             if is_full_context:
#                 # Full context logic
#                 stage2_output = (
#                     cleaned_df.filter(pl.col("document_url") == doc["document_url"][0])
#                     .select(pl.col("document_text"))
#                     .collect()
#                     .to_series()
#                     .to_list()[0]
#                 )
#                 context_token_count = count_tokens_approximate(stage2_output, tokenizer)
#                 retrieval_performance_results = None
#             else:
#                 # RAG logic - your existing retrieval code
#                 if retrieval_kind == "basic_rag":
#                     stage1_output = await base_retrieval(
#                         query=question, k=k, collection=collection, dimensions=EMBEDDING_DIMENSION, retries=retries
#                     )
#                 elif retrieval_kind == "enhanced_rag":
#                     stage1_output = (
#                         await dual_retrieval.retrieve_dual(
#                             question, k=k, collection=collection, use_async=True, dimensions=EMBEDDING_DIMENSION
#                         )
#                     ).combined_results

#                 if rerank:
#                     stage2_output = reranker.rerank_results(query=question, retrieval_results=stage1_output)
#                 else:
#                     stage2_output = stage1_output

#                 # Calculate retrieval metrics
#                 k_range = [x for x in [1, 5, 10, 20, 50, 100] if x <= k]
#                 retrieval_performance_results = retrieval_engine.calculate_retrieval_metrics(
#                     retrieved_chunks=stage2_output,
#                     ground_truth_chunk_ids=[ground_truth_chunk_id],
#                     k_values=k_range,
#                 )
#                 context_token_count = count_tokens_approximate(
#                     extract_context_from_stage_output(stage2_output), tokenizer
#                 )

#             # Generate answers and evaluate (your existing logic)
#             rag_prompt, context_section = create_rag_prompt(query=question, retrieved_contexts=stage2_output)

#             for i in range(iterations_per_question):
#                 generated_answer, input_tokens = await llm_call_with_error_tracking(
#                     prompt=rag_prompt,
#                     model=config.small_llm,
#                     max_tokens=2000,
#                     temperature=0.1,
#                     retries=retries,
#                     client_async=client_async,
#                     model_interface=model_interface,
#                 )

#                 judge_result = await judge.evaluate_answer(
#                     question=question,
#                     expected_answer=ground_truth,
#                     model_answer=generated_answer,
#                     context_preview=rag_prompt,
#                     max_retries=retries,
#                 )

#                 if not judge_result:
#                     logger.warning(f"Judge evaluation failed for question {q_index}, iteration {i}")
#                     continue

#                 question_results.append(
#                     {
#                         "question_index": q_index,  # Add explicit question index for tracking
#                         "k": k,
#                         "retrieval_kind": retrieval_kind,
#                         "rerank": rerank,
#                         "n_question": n_questions,
#                         "judge_result": judge_result,
#                         "retrieval_benchmarks": retrieval_performance_results,
#                         "context_token_count": context_token_count,
#                         "generated_answer": generated_answer,
#                         "stage_2_output": stage2_output if not is_full_context else "full_context_used",
#                         "iteration": i,
#                         "retrieved_chunks": len(stage2_output) if isinstance(stage2_output, list) else 1,
#                         **doc,
#                     }
#                 )

#             # Save checkpoint after each question (atomic operation)
#             if question_results:  # Only save if we have results
#                 try:
#                     await checkpoint_manager.save_question_checkpoint(experiment_id, q_index, question_results)
#                     logger.debug(f"Checkpointed question {q_index} with {len(question_results)} results")
#                 except Exception as checkpoint_error:
#                     logger.error(f"Failed to save checkpoint for question {q_index}: {checkpoint_error}")
#                     # Don't raise - continue with other questions but log the failure
#                     # The final experiment status will show if not all questions were saved

#         # Load all results and finalize experiment
#         all_results = await checkpoint_manager.load_checkpoints(experiment_id)

#         if all_results:
#             try:
#                 final_output_path = await checkpoint_manager.finalize_experiment(experiment_id, all_results)
#                 logger.info(f"Experiment completed successfully: {final_output_path}")
#             except Exception as finalize_error:
#                 logger.error(f"Failed to finalize experiment {experiment_id}: {finalize_error}")
#                 await checkpoint_manager._update_experiment_status(experiment_id, "failed")
#                 raise
#         else:
#             logger.warning(f"No results found for experiment {experiment_id}")
#             await checkpoint_manager._update_experiment_status(experiment_id, "failed")

#         return all_results

#     except Exception as e:
#         # Mark experiment as failed with proper error handling
#         try:
#             await checkpoint_manager._update_experiment_status(experiment_id, "failed")
#         except Exception as status_error:
#             logger.error(f"Failed to update experiment status to failed: {status_error}")

#         logger.error(f"Experiment {experiment_id} failed: {e}")
#         raise


# Integration function that uses your existing notebook variables
async def run_single_experiment_improved(
    qa_pairs,
    cleaned_df,  # Your polars LazyFrame
    k: int = 3,
    iterations_per_question: int = 3,
    retrieval_kind: str = "basic_rag",
    *,
    rerank: bool = True,
    n_questions: int | None = None,
    force_new: bool = False,
    retries: int = 3,
):
    """
    Improved version of your run_single_experiment function with safe checkpointing.
    Uses existing notebook variables: chroma_client, tokenizer, etc.
    """

    checkpoint_manager = SafeCheckpointManager()
    n_questions = n_questions or len(qa_pairs)

    # Start or resume experiment safely
    experiment_id, is_resume, is_completed = await checkpoint_manager.start_experiment(
        k=k, retrieval_kind=retrieval_kind, rerank=rerank, n_questions=n_questions, force_new=force_new
    )

    # If experiment is already completed, return cached results immediately
    if is_completed:
        logger.info(f"Experiment {experiment_id} already completed, loading cached results")
        return await checkpoint_manager.load_completed_experiment_results(experiment_id)

    if is_resume:
        # Get completed questions to skip
        completed_questions = await checkpoint_manager.get_completed_questions(experiment_id)
        logger.info(f"Resuming experiment {experiment_id}: {len(completed_questions)} questions already completed")
    else:
        completed_questions = set()
        logger.info(f"Starting new experiment {experiment_id}")

    try:
        # Pre-filter questions to process (more efficient than checking in loop)
        questions_to_process = [
            (q_index, doc) for q_index, doc in enumerate(qa_pairs[:n_questions])
            if q_index not in completed_questions
            and doc.get("probability_human", 0) >= 0.8
        ]

        logger.info(f"Processing {len(questions_to_process)} questions (skipping {len(completed_questions)} completed)")

        # Process questions with checkpoint recovery
        for q_index, doc in tqdm(questions_to_process):

            # Get collection for this document
            collection = chroma_client.get_collection(f"wikiqa_{doc['collection_suffix']}")
            if collection.count() == 0:
                logger.warning(f"Collection {collection.name} is empty, skipping question {q_index}")
                continue

            question = doc["generated_question"]
            ground_truth = doc["answer"]
            ground_truth_chunk_id = doc["main_chunk_id"]

            # Your existing experiment logic here
            question_results = []

            # Process this question with your existing logic
            is_full_context = retrieval_kind == "full_context"

            if is_full_context:
                # Full context logic
                stage2_output = (
                    cleaned_df.filter(pl.col("document_url") == doc["document_url"][0])
                    .select(pl.col("document_text"))
                    .collect()
                    .to_series()
                    .to_list()[0]
                )
                context_token_count = count_tokens_approximate(stage2_output, tokenizer)
                retrieval_performance_results = None
            else:
                # RAG logic - your existing retrieval code
                if retrieval_kind == "basic_rag":
                    stage1_output = await base_retrieval(
                        query=question, k=k, collection=collection, dimensions=EMBEDDING_DIMENSION, retries=retries
                    )
                elif retrieval_kind == "enhanced_rag":
                    stage1_output = (
                        await dual_retrieval.retrieve_dual(
                            question, k=k, collection=collection, use_async=True, dimensions=EMBEDDING_DIMENSION
                        )
                    ).combined_results

                if rerank:
                    stage2_output = reranker.rerank_results(query=question, retrieval_results=stage1_output)
                else:
                    stage2_output = stage1_output

                # Calculate retrieval metrics
                k_range = [x for x in [1, 5, 10, 20, 50, 100] if x <= k]
                retrieval_performance_results = retrieval_engine.calculate_retrieval_metrics(
                    retrieved_chunks=stage2_output,
                    ground_truth_chunk_ids=[ground_truth_chunk_id],
                    k_values=k_range,
                )
                context_token_count = count_tokens_approximate(
                    extract_context_from_stage_output(stage2_output), tokenizer
                )

            # Generate answers and evaluate (your existing logic)
            rag_prompt, context_section = create_rag_prompt(query=question, retrieved_contexts=stage2_output)

            for i in range(iterations_per_question):
                generated_answer, input_tokens = await llm_call_with_error_tracking(
                    prompt=rag_prompt, model=config.small_llm, max_tokens=2000, temperature=0.1, retries=retries, client_async=client_async, model_interface=model_interface
                )

                judge_result = await judge.evaluate_answer(
                    question=question,
                    expected_answer=ground_truth,
                    model_answer=generated_answer,
                    context_preview=rag_prompt,
                    max_retries=retries,
                )

                if not judge_result:
                    logger.warning(f"Judge evaluation failed for question {q_index}, iteration {i}")
                    continue

                question_results.append(
                    {
                        "question_index": q_index,  # Add explicit question index for tracking
                        "k": k,
                        "retrieval_kind": retrieval_kind,
                        "rerank": rerank,
                        "n_question": n_questions,
                        "judge_result": judge_result,
                        "retrieval_benchmarks": retrieval_performance_results,
                        "context_token_count": context_token_count,
                        "generated_answer": generated_answer,
                        "stage_2_output": stage2_output if not is_full_context else "full_context_used",
                        "iteration": i,
                        "retrieved_chunks": len(stage2_output) if isinstance(stage2_output, list) else 1,
                        **doc,
                    }
                )

            # Save checkpoint after each question (atomic operation)
            if question_results:  # Only save if we have results
                await checkpoint_manager.save_question_checkpoint(experiment_id, q_index, question_results)
                logger.debug(f"Checkpointed question {q_index} with {len(question_results)} results")

        # Load all results and finalize experiment
        all_results = await checkpoint_manager.load_checkpoints(experiment_id)

        if all_results:
            final_output_path = await checkpoint_manager.finalize_experiment(experiment_id, all_results)
            logger.info(f"Experiment completed successfully: {final_output_path}")
        else:
            logger.warning(f"No results found for experiment {experiment_id}")

        return all_results

    except Exception as e:
        # Mark experiment as failed
        await checkpoint_manager._update_experiment_status(experiment_id, "failed")
        logger.error(f"Experiment {experiment_id} failed: {e}")
        raise

async def test_checkpoint_system():
    """Quick test of the checkpoint system with your existing notebook setup."""

    # Test with just one question
    test_results = await run_single_experiment_improved(
        qa_pairs=filtering_results[:2],  # Just 2 questions for testing
        cleaned_df=html_cleaned_df,
        k=5,
        retrieval_kind="enhanced_rag",
        rerank=False,
        n_questions=2,
        force_new=False,  # Force new to test fresh creation
        iterations_per_question=1,  # Just one iteration for speed
    )

    print(f"✅ Test completed: {len(test_results)} results")
    return test_results


test_results = await test_checkpoint_system()
test_results

2025-09-20 15:51:52 | INFO     | improved_checkpoint_system:start_experiment:363 - Found completed experiment but insufficient data (0 < 1.6)
2025-09-20 15:51:52 | INFO     | improved_checkpoint_system:start_experiment:376 - Started new experiment: 5_enhanced_rag_False_2_1758376312_f6658091
2025-09-20 15:51:52 | INFO     | __main__:run_single_experiment_improved:298 - Starting new experiment 5_enhanced_rag_False_2_1758376312_f6658091
2025-09-20 15:51:52 | INFO     | __main__:run_single_experiment_improved:308 - Processing 2 questions (skipping 0 completed)


  0%|          | 0/2 [00:00<?, ?it/s]

2025-09-20 15:52:11 | INFO     | __main__:run_single_experiment_improved:417 - Experiment completed successfully: /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/5_enhanced_rag_False_2_1758376312_f6658091.parquet


✅ Test completed: 2 results


[{'question_index': 0,
  'k': 5,
  'retrieval_kind': 'enhanced_rag',
  'rerank': False,
  'n_question': 2,
  'judge_result': {'absolute_assessment': {'is_correct': True,
    'confidence': 1.0,
    'reasoning': "The model's answer matches the expected answer by stating that Neil Sandilands joined the main cast of The Flash in season 4 alongside the returning principal cast members. It also names the returning cast members, which is supplementary information but does not contradict or misrepresent the core answer."},
   'context_grounded_assessment': {'is_correct_given_context': True,
    'confidence': 1.0,
    'reasoning': "The provided contexts (chunks 1, 2, and 3) clearly demonstrate that Neil Sandilands joined the principal cast, listing him alongside the returning main cast members. The model's answer accurately reflects this information, and all named cast members are explicitly listed in the 'Starring' section in the context."},
   'answer_completeness': 1.0,
   'context_utilizati

In [62]:

async def run_full_experiment_improved(
    clean_df,
    k_range: list[int] | None = None,
    n_questions: int | None = None,
    parallel: int = 3,  # Reduced from 10 to prevent overwhelming the system
    iterations_per_question: int = 3,
    retries: int = 3,
    *,
    force_new: bool = False,
):
    """
    Improved version of your run_full_experiment with better parallelism and checkpointing.
    """
    from functools import reduce
    from itertools import product

    sem = asyncio.Semaphore(parallel)
    k_range = k_range or [1, 5, 10, 50, 100]
    n_questions = n_questions or len(filtering_results)

    # Generate all experiment combinations
    all_combinations = list(product(k_range, ["basic_rag", "enhanced_rag"], [True, False]))
    all_combinations.append((1, "full_context", False))  # Add full context experiment

    async def run_with_semaphore(k, retrieval_kind, rerank):
        async with sem:
            return await run_single_experiment_improved(
                qa_pairs=filtering_results,
                cleaned_df=clean_df,
                k=k,
                retrieval_kind=retrieval_kind,
                rerank=rerank,
                n_questions=n_questions,
                iterations_per_question=iterations_per_question,
                retries=retries,
                force_new=force_new,
            )

    # Create tasks for all experiments
    tasks = []
    # incomplete_experiments = [
    #     # K=1 configurations
    #     (1, "basic_rag", False),
    #     (1, "basic_rag", True),
    #     (1, "enhanced_rag", False),
    #     (1, "enhanced_rag", True),

    #     # K=5 configurations  
    #     (5, "basic_rag", False),
    #     (5, "basic_rag", True),
    #     (5, "enhanced_rag", False),
    #     (5, "enhanced_rag", True),

    #     # K=10 configurations
    #     (10, "basic_rag", False),
    #     (10, "basic_rag", True),
    #     (10, "enhanced_rag", False),
    #     (10, "enhanced_rag", True),

    #     # K=50 configurations
    #     (50, "basic_rag", False),
    #     (50, "basic_rag", True),
    #     (50, "enhanced_rag", False),

    #     # K=100 configuration
    #     (100, "enhanced_rag", True),
    # ]
    # incomplete_experiments = [
    #     # Full context experiments (context window vs RAG comparison)
    #     (1, "full_context", False),
    #     (5, "full_context", False),
    #     (10, "full_context", False),
    #     (20, "full_context", False),
    #     (50, "full_context", False),

    #     # K=20 moderate retrieval experiments
    #     # (20, "basic_rag", False),
    #     (20, "basic_rag", True),
    #     (20, "enhanced_rag", False),
    #     (20, "enhanced_rag", True),

    #     # K=50 high retrieval experiments
    #     (50, "basic_rag", True),
    #     (50, "enhanced_rag", False),
    # ]
    # all_combinations = [
    #     # Full context experiments (context window vs RAG comparison)
    #     (5, "enhanced_rag", False),
    # ]


    logger.info(f"Running {len(all_combinations)} experiments with {n_questions} questions each")
    logger.info(f"Parallelism: {parallel}, Total expected results: {len(all_combinations) * n_questions}")

    for k, retrieval_kind, rerank in all_combinations:
        logger.info(f"Queuing experiment: k={k}, retrieval_kind={retrieval_kind}, rerank={rerank}, n_questions={n_questions}")
        tasks.append(run_with_semaphore(k, retrieval_kind, rerank))

    # Run all experiments with progress tracking
    from tqdm.asyncio import tqdm_asyncio

    results = await tqdm_asyncio.gather(*tasks, desc="Running experiments")

    # Combine all results
    all_results = []
    for result in results:
        if result:  # Skip empty results
            all_results.extend(result)

    logger.info(f"All experiments completed: {len(all_results)} total results")
    return all_results

results = await run_full_experiment_improved(
    clean_df=html_cleaned_df,
    k_range=[100, 200],
    n_questions=200,
    parallel=5,
    retries=6,
)
# results = await run_full_experiment_improved(k_range=[5], n_questions=200, parallel=7, retries=5, clean_df=html_cleaned_df)
# dumped_results = [flatten(x) for x in dump_pydantic_models(results)]
# json.dump(dumped_results, open('../results/reranking_value/wikitext_rag_experiments_results.json', 'w+'), indent=4)
error_summary = summarize_error_statistics(results)
results

2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:92 - Running 9 experiments with 200 questions each
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:93 - Parallelism: 5, Total expected results: 1800
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:96 - Queuing experiment: k=100, retrieval_kind=basic_rag, rerank=True, n_questions=200
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:96 - Queuing experiment: k=100, retrieval_kind=basic_rag, rerank=False, n_questions=200
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:96 - Queuing experiment: k=100, retrieval_kind=enhanced_rag, rerank=True, n_questions=200
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:96 - Queuing experiment: k=100, retrieval_kind=enhanced_rag, rerank=False, n_questions=200
2025-09-21 11:04:57 | INFO     | __main__:run_full_experiment_improved:96 - Queuing experiment: k=200, retrieval_kind

  0%|          | 0/189 [00:00<?, ?it/s]

2025-09-21 11:04:57 | INFO     | improved_checkpoint_system:start_experiment:376 - Started new experiment: 100_enhanced_rag_False_200_1758445497_1c9aee1a
2025-09-21 11:04:57 | INFO     | __main__:run_single_experiment_improved:298 - Starting new experiment 100_enhanced_rag_False_200_1758445497_1c9aee1a
2025-09-21 11:04:57 | INFO     | __main__:run_single_experiment_improved:308 - Processing 189 questions (skipping 0 completed)


  0%|          | 0/189 [00:00<?, ?it/s]

2025-09-21 11:04:57 | INFO     | improved_checkpoint_system:start_experiment:376 - Started new experiment: 100_enhanced_rag_True_200_1758445497_944c4f4e
2025-09-21 11:04:57 | INFO     | __main__:run_single_experiment_improved:298 - Starting new experiment 100_enhanced_rag_True_200_1758445497_944c4f4e
2025-09-21 11:04:57 | INFO     | __main__:run_single_experiment_improved:308 - Processing 189 questions (skipping 0 completed)


...

2025-09-21 15:49:18 | INFO     | __main__:run_single_experiment_improved:295 - Resuming experiment 5_enhanced_rag_True_200_1758029362_32fd4aaf: 32 questions already completed
2025-09-21 15:49:18 | INFO     | __main__:run_single_experiment_improved:308 - Processing 157 questions (skipping 32 completed)


  0%|          | 0/157 [00:00<?, ?it/s]

Running experiments:   0%|          | 0/5 [22:21<?, ?it/s]


CancelledError: 

2025-09-21 16:14:32 | WARNING  | context_is_king.rag_pipeline.query_enhancement.dual_retrieval:generate_hypothetical_answer:66 - Hypothetical answer generation failed
2025-09-21 16:17:19 | WARNING  | context_is_king.rag_pipeline.query_enhancement.dual_retrieval:generate_hypothetical_answer:66 - Hypothetical answer generation failed
2025-09-21 16:22:35 | ERROR    | __main__:llm_call_with_error_tracking:86 - Bad request error (content policy), not retrying: Error code: 400 - {'code': 400, 'error': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'source': 'provider'}
2025-09-21 16:22:40 | ERROR    | __main__:llm_call_with_error_tracking:86 - Bad request error (content policy), not retrying: Error code: 400 - {'code': 400, 'error': "The response was filtered due to the promp

In [63]:
tmp = pl.read_parquet('/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/100_enhanced_rag_False_200_1758445497_1c9aee1a.parquet')
tmp.shape

(567, 49)

In [64]:
tmp.select(pl.col('k').value_counts())

k
struct[2]
"{100,567}"


In [22]:
from context_is_king import DATA_DIR
tmpdf = pl.DataFrame(results).unique()
print(f'Loaded {len(tmpdf)} datapoints from results')
tmpdf.write_parquet(DATA_DIR / '5_enhanced_rag_200_results.parquet')

Loaded 566 datapoints from results


In [57]:
schema = {
      'question_index': pl.Int64,
      'k': pl.Int64,
      'retrieval_kind': pl.String,
      'rerank': pl.Boolean,
      'n_question': pl.Int64,
      'judge_result_absolute_assessment_is_correct': pl.Boolean,
      'judge_result_absolute_assessment_confidence': pl.Float64,
      'judge_result_absolute_assessment_reasoning': pl.String,
      'judge_result_context_grounded_assessment_is_correct_given_context': pl.Boolean,
      'judge_result_context_grounded_assessment_confidence': pl.Float64,
      'judge_result_context_grounded_assessment_reasoning': pl.String,
      'judge_result_answer_completeness': pl.Float64,
      'judge_result_context_utilization': pl.Float64,
      'retrieval_benchmarks_precision@1': pl.Float64,
      'retrieval_benchmarks_recall@1': pl.Float64,
      'retrieval_benchmarks_f1@1': pl.Float64,
      'retrieval_benchmarks_precision@5': pl.Float64,
      'retrieval_benchmarks_recall@5': pl.Float64,
      'retrieval_benchmarks_f1@5': pl.Float64,
      'retrieval_benchmarks_precision@10': pl.Float64,
      'retrieval_benchmarks_recall@10': pl.Float64,
      'retrieval_benchmarks_f1@10': pl.Float64,
      'retrieval_benchmarks_mrr': pl.Float64,
      'retrieval_benchmarks_hit_rate': pl.Float64,
      'retrieval_benchmarks_avg_similarity': pl.Float64,
      'retrieval_benchmarks_max_similarity': pl.Float64,
      'retrieval_benchmarks_min_similarity': pl.Float64,
      'context_token_count': pl.Int64,
      'generated_answer': pl.String,
      'stage_2_output': pl.List(pl.Struct({
          'chunk': pl.Struct({
              'content': pl.String,
              'doc_id': pl.String,
              'chunk_id': pl.String,
              'start_char': pl.Int64,
              'end_char': pl.Int64,
              'metadata': pl.Struct({
                  'document_url': pl.String,
                  'collection_name': pl.String,
                  'cache_name': pl.String
              }),
              'full_id': pl.String
          }),
          'similarity_score': pl.Float64,
          'rank': pl.Int64,
          'reranked': pl.Boolean,
          'rerank_score': pl.Float64,
          'final_score': pl.Float64
      })),
      'iteration': pl.Int64,
      'retrieved_chunks': pl.Int64,
      'probability_human': pl.Float64,
      'generated_question': pl.String,
      'question_reasoning': pl.String,
      'main_chunk_id': pl.String,
      'answer': pl.String,
      'status': pl.String,
      'document_url': pl.List(pl.String),
      'collection_suffix': pl.String,
      'error_occurred': pl.Boolean,
      'error_type': pl.String,
      'error_message': pl.String,
      'error_code': pl.String,
      'content_policy_violation': pl.Boolean,
      'input_tokens': pl.Int64,
  }

In [60]:
from pathlib import Path

wiki_folder = (DATA_DIR / '.cache/wikiqa')
wiki_files = list(wiki_folder.iterdir())
for f in tqdm(wiki_files):
    if f.is_dir():
        jsons_in_dir = list(f.glob('*.json'))
        if len(jsons_in_dir) < 5:
            for file_path in jsons_in_dir:
                file_path.unlink()
            f.rmdir()
        print(f'Found {len(jsons_in_dir)} files for {f}')
        target_path = f / 'total_experiment_output.parquet'
        # print(pl.read_json(jsons_in_dir[0]).schema)
        if len(jsons_in_dir) > 50 and '_200_' in f.name:
            data = [json.load(x.open()) for x in jsons_in_dir if x.suffix == '.json']
            data = [flatten(x) for y in data for x in y]
            for x in data:
                x.pop('stage_2_output')
            dir_df = pl.DataFrame(data, schema=schema).with_columns(pl.col('document_url').list.first())
            # dir_df = pl.concat([pl.read_json(x) for x in jsons_in_dir if x.suffix == '.json'])
            print(f'Loaded {len(dir_df)} rows from jsons.')
            # if len(dir_df) != 3*len(jsons_in_dir):
            #     choice = input(f'{len(dir_df)=} != {len(jsons_in_dir)}. Continue writing to disk?')
            #     if 'y' in choice.lower():
            #         dir_df.write_parquet(f / 'total_experiment_output.parquet')
            #     continue
            print(f'Writing output to {target_path}')
            dir_df.write_parquet(target_path)
        else: 
            target_path.unlink()


  0%|          | 0/234 [00:00<?, ?it/s]

Found 186 files for /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/10_enhanced_rag_True_200_1758019334_3c586b8a_checkpoints
Loaded 558 rows from jsons.
Writing output to /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/10_enhanced_rag_True_200_1758019334_3c586b8a_checkpoints/total_experiment_output.parquet
Found 156 files for /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/50_enhanced_rag_False_200_1758048375_0ebff9c0_checkpoints
Loaded 468 rows from jsons.
Writing output to /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/50_enhanced_rag_False_200_1758048375_0ebff9c0_checkpoints/total_experiment_output.parquet
Found 28 files for /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/50_basic_rag_True_200_1758048508_49d6068b_checkpoints
Found 103 files for /Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-k

In [61]:
pl.scan_parquet(DATA_DIR / '**/total_experiment_output.parquet').select(pl.len()).collect()

len
u32
6819


In [20]:
tmpdf['iteration'].value_counts()

iteration,count
i64,u32
1,189
2,188
0,189


In [23]:
Path('../data/.cache/wikiqa/batch2_results.json').resolve()

PosixPath('/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/batch2_results.json')

In [28]:
Path('../data/.cache/wikiqa').resolve().exists()

True

In [29]:
json.dump(results, open('../data/.cache/wikiqa/batch2_results.json', 'w+'), )

In [20]:
import pandas as pd
pd.DataFrame(results).to_parquet('../data/.cache/wikiqa/batch2results.parquet')

ArrowTypeError: ("Expected bytes, got a 'list' object", 'Conversion failed for column stage_2_output with type object')

In [30]:
pl.DataFrame(results, infer_schema_length=1000, strict=False)

ComputeError: could not append value: [{{"The Flash (season 4) - Wikipedia

 

The Flash (season 4) 
From Wikipedia, the free encyclopedia 

					Jump to:					navigation, 					search

The Flash (season 4)

Promotional poster

Starring

Grant Gustin
Candice Patton
Danielle Panabaker
Carlos Valdes
Keiynan Lonsdale
Neil Sandilands
Tom Cavanagh
Jesse L. Martin

Country of origin
United States

No. of episodes
19

Release

Original network
The CW

Original release
October 10, 2017 (2017-10-10) – present (present)

Season chronology

← Previous
Season 3

List of The Flash episodes

The fourth season of the American television series The Flash, which is based on the DC Comics character Barry Allen / Flash, follows a crime scene investigator with superhuman speed who fights criminals, including others who have also gained superhuman abilities. It is set in the Arrowverse, sharing continuity with the other television series of the universe, and is a spin-off of Arrow. The season is produced by 

In [50]:
list(Path('../data/.cache/wikiqa').glob('*.parquet'))

[PosixPath('../data/.cache/wikiqa/50_enhanced_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/50_enhanced_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/5_basic_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/5_enhanced_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/20_enhanced_rag_True_1.parquet'),
 PosixPath('../data/.cache/wikiqa/5_enhanced_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/20_basic_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/20_enhanced_rag_True_20.parquet'),
 PosixPath('../data/.cache/wikiqa/20_enhanced_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/1_basic_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/10_basic_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/5_basic_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/20_basic_rag_False_200.parquet'),
 PosixPath('../data/.cache/wikiqa/50_basic_rag_True_200.parquet'),
 PosixPath('../data/.cache/wikiqa/100_basic

In [75]:
import glob


target_struct = pl.Struct(
    {
        "chunk": pl.Struct(
            {
                "content": pl.String,
                "doc_id": pl.String,
                "chunk_id": pl.String,
                "start_char": pl.Int64,
                "end_char": pl.Int64,
                "metadata": pl.Struct(
                    {"document_url": pl.String, "cache_name": pl.String, "collection_name": pl.String}
                ),
                "full_id": pl.String,
            }
        ),
        "similarity_score": pl.Float64,
        "rank": pl.Int64,
        "reranked": pl.Boolean,
        "rerank_score": pl.Float64,
        "final_score": pl.Float64,
    }
)

dfs = []
cast_cols = ["retrieval_benchmarks_f1@5"]
for file in glob.glob("../data/.cache/wikiqa/*.parquet"):
    df = pl.scan_parquet(file)
    # print(file)
    schema = df.collect_schema()
    for c in cast_cols:
        if c in schema.names():
            df = df.with_columns(pl.col(c).cast(pl.Float64))
    if 'stage_2_output' in schema:
        if schema['stage_2_output'] == pl.String:
            # Only strings get converted to empty lists
            df = df.with_columns(
                pl.lit([]).cast(pl.List(target_struct)).alias('stage_2_output')
            )
    # df = df.with_columns(
    #     pl.when(pl.col("stage_2_output") == "full_context_used")
    #     .then(pl.lit([]).cast(pl.List(pl.Struct)))
    #     .otherwise(pl.col("stage_2_output"))
    #     .alias("stage3")
    # )
    dfs.append(df)
    # break
    # df = df.with_columns(pl.col('stage2_output').replace())
results_df = pl.concat(dfs, how="diagonal")

# results_df.sink_parquet('../data/processed/wikiqa/wikiqa_full_results.parquet', mkdir=True)
results_df

InvalidOperationError: 'union'/'concat' inputs should all have the same schema,got
Schema:
name: k, field: Int64
name: retrieval_kind, field: String
name: rerank, field: Boolean
name: n_question, field: Int64
name: judge_result_absolute_assessment_is_correct, field: Boolean
name: judge_result_absolute_assessment_confidence, field: Float64
name: judge_result_absolute_assessment_reasoning, field: String
name: judge_result_context_grounded_assessment_is_correct_given_context, field: Boolean
name: judge_result_context_grounded_assessment_confidence, field: Float64
name: judge_result_context_grounded_assessment_reasoning, field: String
name: judge_result_answer_completeness, field: Float64
name: judge_result_context_utilization, field: Float64
name: retrieval_benchmarks_precision@1, field: Float64
name: retrieval_benchmarks_recall@1, field: Float64
name: retrieval_benchmarks_f1@1, field: Int64
name: retrieval_benchmarks_precision@5, field: Float64
name: retrieval_benchmarks_recall@5, field:

In [79]:
results_df = pl.read_parquet('../data/processed/wikiqa/wikiqa_full_results.parquet')
results_df

k,retrieval_kind,rerank,n_question,judge_result_absolute_assessment_is_correct,judge_result_absolute_assessment_confidence,judge_result_absolute_assessment_reasoning,judge_result_context_grounded_assessment_is_correct_given_context,judge_result_context_grounded_assessment_confidence,judge_result_context_grounded_assessment_reasoning,judge_result_answer_completeness,judge_result_context_utilization,retrieval_benchmarks_precision@1,retrieval_benchmarks_recall@1,retrieval_benchmarks_f1@1,retrieval_benchmarks_precision@5,retrieval_benchmarks_recall@5,retrieval_benchmarks_f1@5,retrieval_benchmarks_precision@10,retrieval_benchmarks_recall@10,retrieval_benchmarks_f1@10,retrieval_benchmarks_precision@20,retrieval_benchmarks_recall@20,retrieval_benchmarks_f1@20,retrieval_benchmarks_precision@50,retrieval_benchmarks_recall@50,retrieval_benchmarks_f1@50,retrieval_benchmarks_mrr,retrieval_benchmarks_hit_rate,retrieval_benchmarks_avg_similarity,retrieval_benchmarks_max_similarity,retrieval_benchmarks_min_similarity,context_token_count,generated_answer,iteration,retrieved_chunks,probability_human,generated_question,question_reasoning,main_chunk_id,answer,status,document_url,collection_suffix,stage_2_approach,retrieval_benchmarks_precision@100,retrieval_benchmarks_recall@100,retrieval_benchmarks_f1@100,retrieval_benchmarks,stage_2_output
i64,str,bool,i64,bool,f64,str,bool,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,i64,i64,f64,str,str,str,str,str,str,str,str,f64,f64,f64,null,str
50,"""enhanced_rag""",false,200,true,0.95,"""The model's answer matches the…",true,1.0,"""The answer is strongly support…",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021185,0.032787,0.010417,15073,"""As of July 2017, the following…",0,50,0.85,"""Which states have enacted stat…","""The text details which states …","""6c357748-1654-46c5-9798-82cc3e…","""""","""question_generation_succeeded""","""https://en.wikipedia.org//w/in…","""List_of_smoking_bans_in_the_Un…","""rag_retrieval""",null,null,null,null,null
50,"""enhanced_rag""",false,200,true,0.95,"""The model's answer accurately …",true,0.95,"""The model used the context ext…",0.95,0.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021185,0.032787,0.010417,15073,"""The states that have enacted s…",1,50,0.85,"""Which states have enacted stat…","""The text details which states …","""6c357748-1654-46c5-9798-82cc3e…","""""","""question_generation_succeeded""","""https://en.wikipedia.org//w/in…","""List_of_smoking_bans_in_the_Un…","""rag_retrieval""",null,null,null,null,null
50,"""enhanced_rag""",false,200,true,0.95,"""The model's answer matches the…",true,1.0,"""The model's answer is strictly…",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021185,0.032787,0.010417,15073,"""The following states have enac…",2,50,0.85,"""Which states have enacted stat…","""The text details which states …","""6c357748-1654-46c5-9798-82cc3e…","""""","""question_generation_succeeded""","""https://en.wikipedia.org//w/in…","""List_of_smoking_bans_in_the_Un…","""rag_retrieval""",null,null,null,null,null
50,"""enhanced_rag""",true,200,false,1.0,"""The ground truth states that M…",true,1.0,"""Based on the provided context,…",0.7,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.020972,0.032258,0.011364,22371,"""I don't have enough informatio…",0,50,0.8,"""What abilities does Mina Chayt…","""This question is appropriate b…","""b32b59e0-0e5f-4513-ae4b-afc053…","""Mina Chayton can animate statu…","""question_generation_succeeded""","""https://en.wikipedia.org//w/in…","""The_Flash_season_4""","""rag_retrieval""",null,null,null,null,null
50,"""enhanced_rag""",true,200,false,1.0,"""The ground truth answer is tha…",true,1.0,"""Upon reviewing all the provide…",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.020972,0.032258,0.011364,22371,"""I don't have e

In [18]:
import altair as alt
import polars as pl

In [20]:
results_df.columns

['k',
 'enhanced_rag',
 'rerank',
 'n_question',
 'judge_result_absolute_assessment_is_correct',
 'judge_result_absolute_assessment_confidence',
 'judge_result_absolute_assessment_reasoning',
 'judge_result_context_grounded_assessment_is_correct_given_context',
 'judge_result_context_grounded_assessment_confidence',
 'judge_result_context_grounded_assessment_reasoning',
 'judge_result_answer_completeness',
 'judge_result_context_utilization',
 'retrieval_benchmarks_precision@1',
 'retrieval_benchmarks_recall@1',
 'retrieval_benchmarks_f1@1',
 'retrieval_benchmarks_mrr',
 'retrieval_benchmarks_hit_rate',
 'retrieval_benchmarks_avg_similarity',
 'retrieval_benchmarks_max_similarity',
 'retrieval_benchmarks_min_similarity',
 'generated_answer',
 'stage_1_output',
 'stage_2_output',
 'iteration',
 'human_would_ask',
 'generated_question',
 'question_reasoning',
 'main_chunk_id',
 'answer',
 'status',
 'document_url',
 'collection_suffix']

In [22]:
results_df.schema

Schema([('k', Int64),
        ('enhanced_rag', Boolean),
        ('rerank', Boolean),
        ('n_question', Int64),
        ('judge_result_absolute_assessment_is_correct', Boolean),
        ('judge_result_absolute_assessment_confidence', Float64),
        ('judge_result_absolute_assessment_reasoning', String),
        ('judge_result_context_grounded_assessment_is_correct_given_context',
         Boolean),
        ('judge_result_context_grounded_assessment_confidence', Float64),
        ('judge_result_context_grounded_assessment_reasoning', String),
        ('judge_result_answer_completeness', Float64),
        ('judge_result_context_utilization', Float64),
        ('retrieval_benchmarks_precision@1', Float64),
        ('retrieval_benchmarks_recall@1', Float64),
        ('retrieval_benchmarks_f1@1', Float64),
        ('retrieval_benchmarks_mrr', Float64),
        ('retrieval_benchmarks_hit_rate', Float64),
        ('retrieval_benchmarks_avg_similarity', Float64),
        ('retrieval_be

In [71]:
results